<a href="https://colab.research.google.com/github/4cekay/B101-Group2-NLP-Project/blob/main/DatasetCuration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setup**

In [1]:
!pip install numpy datasets

In [2]:
import pandas as pd
import glob
import re
from datasets import load_dataset

**Helper #1:** cleaning formatted text (particularly for generated articles):

- Escaped strings
- Special tokens
- Formatting, markdown
- Metadata
- Placeholders
- Footers, promotional plugs

In [3]:
def clean_article(text, max_words=250):
  """
  Cleans article text blocks by removing all noisy instances listed below, and keeping derived sample within max word count.
  """
  if not text:
    return ""

  # if applicable, handle ALL instances of the following:

  # Escaped Strings
  text = text.replace("\\n", "\n")

  # Special Token (i.e. gemma's <end_of_turn>)
  text = re.sub(r"<end_of_turn>", "", text)

  # Formatting
  text = re.sub(r"^\s*#+\s*", "", text, flags=re.MULTILINE)         # header hashes
  text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)                      # bold asterisks
  text = re.sub(r"^\s[\*\-]\s*", "", text, flags=re.MULTILINE)      # bullet points
  text = re.sub(r"^\s*[-*_]{3,}\s*$", "", text, flags=re.MULTILINE) # dividers

  # Metadata
  text = re.sub(r"^\s*(?:#+|\*\*)*\s*title\s*(?:\*\*)?\s*:\s*", "", text, flags=re.IGNORECASE | re.MULTILINE)       # "Title:"
  text = re.sub(r"^\s*(?:#+|\*\*)*\s*by\b.*$", "", text, flags=re.IGNORECASE | re.MULTILINE)                         # "by" writer
  text = re.sub(r"^\s(?:#+|\*\*)*\s*(?:published|date|author)\s*:.*$", "", text, flags=re.IGNORECASE | re.MULTILINE) # publication details
  text = re.sub(r"^\s*(?:#+|\*\*)*\s*background context\s*|(?:\*\*)?\s*:\s*", "", text, flags=re.IGNORECASE | re.MULTILINE)  # "Background Context"


  # Placeholders
  text = re.sub(r"\[\s*(?:insert|your|author|date|link|chart|image)[^\]]*\]", "", text, flags=re.IGNORECASE)

  # Footers
  text = re.sub(r"^\s*(?:#+|\*\*)*\s*(?:published|date|author)\s*:.*$", "", text, flags=re.IGNORECASE | re.MULTILINE)

  # keep # of paragraphs sampled within the word count (split by \n\n)
  paragraphs = [p.strip() for p in re.split(r'\n\s*\n+', text) if p.strip()]
  kept_paragraphs = []
  total_word_count = 0

  for p in paragraphs: # check each paragraph to make sure sample stays within max
    para_word_count = len(p.split())

    if total_word_count + para_word_count <= max_words:
      kept_paragraphs.append(p)
      total_word_count += para_word_count
    else:
      break

  # If the first paragraph is already too long, just don't use the sample
  if not kept_paragraphs:
    return None

  return "\n\n".join(kept_paragraphs)

In [24]:
def clean_CNN(text, max_words=250):
  """
  A revised version of the above function to clean CNN articles ALREADY IN THE SPLIT that are massive noisy blocks of tags, pain, and suffering.
  Attempts to truncate by paragraph, but with the length of some article blocks, sentence-trimming below word count might be necessary.
  """
  if not text:
    return ""

  # Escaped Strings
  text = text.replace("\\n", "\n")

  # Links, Embeds, Labels, etc.
  text = re.sub(r"Follow @\w+", "", text)
  text = re.sub(r"\bRead More\b", "", text, flags=re.IGNORECASE)
  text = re.sub(r"JUST WATCHED.*?Replay", "", text, flags=re.IGNORECASE | re.DOTALL)
  text = re.sub(r"More Videos\s*\.\.\.", "", text, flags=re.IGNORECASE)
  text = re.sub(r"MUST WATCH.*?\d{2}:\d{2}", "", text, flags=re.IGNORECASE | re.DOTALL)
  text = re.sub(r"Visit CNN\.com.*?videos", "", text, flags=re.IGNORECASE | re.DOTALL)

  # Twitter Embeds
  text = re.sub(r"pic\.twitter\.com/\S+", "", text)
  text = re.sub(r"#\w+", "", text)
  text = re.sub(r"[—\-]?\s*[\w\s]+?\(@\w+\)\s*\w+\s*\d{1,2},?\s*\d{4}", "", text)

  # Resulting Whitespace (if any)
  text = re.sub(r"[ \t]+", " ", text).strip()
  text = re.sub(r"\n{3,}", "\n\n", text)

  paragraphs = [p.strip() for p in re.split(r'\n\s*\n+', text) if p.strip()]
  kept_paragraphs = []
  total_word_count = 0

  for p in paragraphs:
    para_words = p.split()
    para_word_count = len(para_words)

    if total_word_count + para_word_count <= max_words:
      kept_paragraphs.append(p)
      total_word_count += para_word_count
    else: # specifically fits sentences within word count to avoid complete removal of samples with long first paragraphs.
      remaining_space = max_words - total_word_count

      if remaining_space > 0:
        truncated_words = para_words[:remaining_space]
        truncated = " ".join(truncated_words)

        last_punc = max( # check for last sentence-ender
            truncated.rfind("."),
            truncated.rfind("!"),
            truncated.rfind("?"),
        )
        # in the case of a long first paragraph with a running sentence, avoid cutting too much.
        if last_punc > len(truncated) * 0.7:
          truncated = truncated[:last_punc + 1]

        kept_paragraphs.append(truncated)
        total_word_count += remaining_space
      break

  # just in case, but hopefully all samples in final set have SOMETHING readable...?
  if not kept_paragraphs:
    return None

  return "\n\n".join(kept_paragraphs)

**Helper #2:** Gathering test/val split from a random subset of a chosen size

In [5]:
def split_subset(samples, subset_size, min_words, max_words, train_split=0.8, column="sample_text"):
  """
  Custom train/val subset splitter that will filter out unviable samples (out of word count range)
  """

  if samples is None or len(samples) ==0:
    return pd.DataFrame(), pd.DataFrame()

  # subset of random samples within set
  subset = (
      samples
      .filter(lambda example: min_words <= len(example[column].split()) <= max_words)
      .shuffle(seed=42)
      .select(range(subset_size))
  )

  # subset to dataframe + clear \n
  df_subset = subset.to_pandas()
  df_subset[column] =  df_subset[column].str.replace(r'[\r\n]+', " ", regex=True).str.strip()

  # split subset into train and val
  n_train = round(len(df_subset) * train_split) # number of training samples

  df_subset_train = df_subset.sample(n=n_train, random_state=42) # get train samples
  df_subset_val = df_subset.drop(df_subset_train.index)          # remove train samples to get val remainder


  return df_subset_train.reset_index(drop=True), df_subset_val.reset_index(drop=True)

# **Extraction**
[HF Dataset Processing](https://huggingface.co/docs/datasets/v1.4.0/processing.html)


## Email Replies

### Human-written

**Stanford Humanual-Email Dataset**

Columns:
- **completion:** ground-truth email reply
  - Main target text sample !!
- post_id (discard)
- user_id (discard)
- timestamp (discard)
- turn_id (discard)
  - Note: In conversation "turn 1" rows, I can take the original email content to use as a prompt for AI-generated samples to compare (w/ proper attribution)
- persona (discard)
- **prompt:** role and content
  - Will pull the initial email content of select conversations to use in prompting later.  
- metadata (discard)


In [ ]:
# Load the raw dataset (https://huggingface.co/datasets/snap-stanford/humanual-email)

raw_dataset = load_dataset('snap-stanford/humanual-email')
raw_dataset


README.md:   0%|          | 0.00/2.71k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  675kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/val-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6377 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/536 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/130 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['completion', 'post_id', 'user_id', 'timestamp', 'turn_id', 'persona', 'prompt', 'metadata'],
        num_rows: 6377
    })
    test: Dataset({
        features: ['completion', 'post_id', 'user_id', 'timestamp', 'turn_id', 'persona', 'prompt', 'metadata'],
        num_rows: 536
    })
    val: Dataset({
        features: ['completion', 'post_id', 'user_id', 'timestamp', 'turn_id', 'persona', 'prompt', 'metadata'],
        num_rows: 130
    })
})

1. I'm gonna extract the initial email content of each conversation thread to use as prompts for our generative email samples later.

In [ ]:
# extracting only the rows with turn_id == 1, indicating the first "turn" in a round of emails

initial_emails = raw_dataset.filter(lambda example: example["turn_id"] == 1)

Filter:   0%|          | 0/6377 [00:00<?, ? examples/s]

Filter:   0%|          | 0/536 [00:00<?, ? examples/s]

Filter:   0%|          | 0/130 [00:00<?, ? examples/s]

In [ ]:
# removing all unwanted features, keeping only the "completion" and "prompt"
initial_emails = initial_emails.remove_columns(["post_id", "user_id", "timestamp", "turn_id", "persona", "metadata"])

2. Now, we take the actual email replies to use as training data.

In [ ]:
email_replies = initial_emails.map(
    lambda example: {
        "sample_replies": example["completion"].strip('\'"')
    },
    remove_columns= ["prompt", "completion"]
)

Map:   0%|          | 0/3780 [00:00<?, ? examples/s]

Map:   0%|          | 0/371 [00:00<?, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

3. Take 900 samples from the train set that are within the word count range.

In [ ]:
max_words = 250
min_words = 20
sample_count = 900

email_replies_subset = (
    email_replies["train"]
    .filter(lambda example: min_words <= len(example["sample_replies"].split()) <= max_words)
    .shuffle(seed=42) # maintain same samples in notebook
    .select(range(sample_count))
)

Filter:   0%|          | 0/3780 [00:00<?, ? examples/s]

In [ ]:
email_replies_subset

Dataset({
    features: ['sample_replies'],
    num_rows: 900
})

In [ ]:
df_email_replies = email_replies_subset.to_pandas()
print(len(email_replies_subset))

df_email_replies

900


,sample_replies
0,"Louise,\n\nWe are happy to do this.\n\nShall w..."
1,"Per Pam, the person we should be talking to is..."
2,"Louise,\n\nAttached is the list of Top 50 coun..."
3,Sounds great. Just check with Audrey (3-5849) ...
4,Sally\n\nProposed list - of all potential cand...
...,...
895,Sorry about my incomplete memo (I hit send ins...
896,I'm a bit confused - I think that the current ...
897,Please see the three tabs for EGS and complete...
898,I maybe mistaken but I think Jack had given me...


In [ ]:
# index=False to get rid of the index no. column

df_email_replies.to_csv('human_email_samples.csv', index=False)

### AI-generated

Gathering the context (prompt --> content) for a number of email replies (completion) in the dataset.
These will be fed into different generative AI models, asking for an appropriate response.

In [ ]:
# extract the email content from each prompt,
# get rid of quotation marks from the beginning and end
# remove the old columns
email_prompts = initial_emails.map(
    lambda example: {
        "text": example["prompt"][0]["content"].strip('\'"') # get rid of quotes
    },
    remove_columns= ["completion", "prompt"]
)

Map:   0%|          | 0/3780 [00:00<?, ? examples/s]

Map:   0%|          | 0/371 [00:00<?, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

In [ ]:
max_words = 250
min_words = 20
sample_count = 225

email_prompts_subset = (
    email_prompts["train"]
    .filter(lambda example: min_words <= len(example["text"].split()) <= max_words)
    .shuffle(seed=42) # maintain same samples in notebook
    .select(range(sample_count))
)

Filter:   0%|          | 0/3780 [00:00<?, ? examples/s]

In [ ]:
df_email_prompts = email_prompts_subset.to_pandas()

df_email_prompts

,text
0,The Ninth Circuit Court of Appeals has schedul...
1,Ken Lay spoke with several California CEOs thi...
2,"TJ,\n\nThe following Realtime employees on the..."
3,Revised Trading Policy. We would like to get ...
4,Legal -\n\nAttached below is an Omnibus Confir...
...,...
220,"Chris,\n\nJust wanted to verify a portion of s..."
221,The issue of O&M fees at new interconnects has...
222,Daren - meter 1517 has a nom of 0/day for Jan....
223,There are about 20-30 deals in pending sent to...


In [ ]:
# csv export example (index=False to get rid of the index no. column)

df_email_prompts.to_csv('human_email_prompts.csv', index=False)

AI-Generated Replies to each prompt email:

Prompt: Can you reply (within 250 words) to each of the 225 emails attached in an appropriate, professional manner?

Export to a csv file with one sample_text column with replies.

[Attached human_email_prompts.csv]

In [ ]:
# splitting the bigger csv into halves for agents (gemini)

email_prompts_1, email_prompts_2 = split_subset(email_prompts_subset, 225, 20, 250, train_split=0.5, column="text")

print(email_prompts_1)
print(email_prompts_2)

Filter:   0%|          | 0/225 [00:00<?, ? examples/s]

                                                  text
0    Mike:  Attached is a spreadsheet that summariz...
1    We apologize about the error in the previous e...
2    True Orange E-Mail/Fax Service Volume 9, E-Mai...
3    Sharon - I need your assistance with respect t...
4    In January 2001, Transwestern had $127,750 rev...
..                                                 ...
107  Ken Lay spoke with several California CEOs thi...
108  Tracy, As you know Project Sunrise no longer h...
109  Phillip, I have attached a list of the Common ...
110  Just found out that we have acces to 15,000 of...
111  Apologies for the late notice!  Please remove ...

[112 rows x 1 columns]
                                                  text
0    Greetings.  Hope, given the circumstances, tha...
1    Revised Trading Policy.  We would like to get ...
2    Rod and Joe, Here is a proposed write up of th...
3    Quck question:  w.r.t. to the stipulation with...
4    Leslie, I just spoke with the gas fu

In [ ]:
email_prompts_1.to_csv("human_email_prompts_112", index=False)
email_prompts_2.to_csv("human_email_prompts_113", index=False)

In [ ]:
from datasets import Dataset

# even smaller splits (copilot)

email_prompts_half_1 = Dataset.from_pandas(email_prompts_1)
email_prompts_half_2 = Dataset.from_pandas(email_prompts_2)

email_prompts_11, email_prompts_12 = split_subset(email_prompts_half_1, 112, 20, 250, train_split=0.5, column="text")
email_prompts_21, email_prompts_22 = split_subset(email_prompts_half_2, 112, 20, 250, train_split=0.5, column="text")

print(email_prompts_11)
print(email_prompts_12)
print(email_prompts_21)
print(email_prompts_22)

Filter:   0%|          | 0/112 [00:00<?, ? examples/s]

Filter:   0%|          | 0/113 [00:00<?, ? examples/s]

                                                 text
0   Columbia Gas Transmission Corporation has in i...
1   Andy: \tI am in the process of revising my for...
2   FIT1 for AR_AP and Earnings products completed...
3   Legal has been asked to review and approve all...
4   Carrie \tMr. Colin Wilkes from General Electri...
5   Michael Moran, currently Managing Director and...
6   Chris, Just wanted to verify a portion of sita...
7   True Orange E-Mail/Fax Service Volume 9, E-Mai...
8   Louise, Attached is a new table derived from t...
9   Darron - Thanks for your continued help with t...
10  Phillip,        I received a fax of the consen...
11  We apologize about the error in the previous e...
12  EFFECTS ON DISTRIBUTED ENERGY ARE UNCLEAR On S...
13  Billy: I hope all is well.  Per our discussion...
14  We will hold off transferring any physical pos...
15  As discussed these individuals will transfer t...
16  John, per your request price for front and bac...
17  I want to let you all kn

In [ ]:
email_prompts_11.to_csv("human_email_prompts_q1_56.csv", index=False)
email_prompts_12.to_csv("human_email_prompts_q2_56.csv", index=False)
email_prompts_21.to_csv("human_email_prompts_q3_56.csv", index=False)
email_prompts_22.to_csv("human_email_prompts_q4_56.csv", index=False)

In [ ]:
email_prompts_21['text'][28]


"whatup beeatch . . . are you definitely gonna go out thurs nite . . . i'm goin out with a chick sometime this week/weekend but haven't decided which nite . . . when are you taking that hot physical therapist out ?"

In [ ]:
path = r"C:\Users\kacey\OneDrive\IAT360\NLP_Project\generated_samples\*.csv"
ai_samples_csv = glob.glob(path)

ai_email_train = []
ai_email_val = []

for file in ai_samples_csv:
  df = pd.read_csv(file)

  # 5 random rows
  val_sample = df.sample(n=5, random_state=42)
  ai_email_val.append(val_sample)

  # rest to training
  train_sample = df.drop(val_sample.index)
  ai_email_train.append(train_sample)

# concat
df_ai_email_train = pd.concat(ai_email_train, ignore_index=True)
df_ai_email_val = pd.concat(ai_email_val, ignore_index=True)

# save to their own csv file
df_ai_email_train.to_csv('train_ai_email_samples.csv', index=False)
df_ai_email_val.to_csv('val_ai_email_samples.csv', index=False)

print(f"Validation rows: {len(df_ai_email_val)}")
print(f"Training rows: {len(df_ai_email_train)}")


Validation rows: 20
Training rows: 80


In [ ]:
path_112 = r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\gemini\gemini_cleaned_1.csv"
path_113 = r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\gemini\gemini_cleaned_2.csv"


df_gemini_112 = pd.read_csv(path_112)
df_gemini_113 = pd.read_csv(path_113)

gemini = [df_gemini_112, df_gemini_113]

df_gemini_complete= pd.concat(gemini, ignore_index=True)

df_gemini_complete.to_csv("gemini_emails_225.csv", index=False)
print(len(df_gemini_complete))

225


In [ ]:
path = r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\generated_samples\emails (by model)\225\*.csv"
ai_samples_csv = glob.glob(path)

ai_emails = []

for file in ai_samples_csv:
  df = pd.read_csv(file)
  ai_emails.append(df)

# concat
df_ai_emails = pd.concat(ai_emails, ignore_index=True)

# save to their own csv file
df_ai_emails.to_csv('ai_emails_900.csv', index=False)

print(len(df_ai_emails))


900


## Abstracts

### AI-generated

In [ ]:
# https://huggingface.co/datasets/Ateeqq/AI-and-Human-Generated-Text

raw_dataset = load_dataset('Ateeqq/AI-and-Human-Generated-Text')
raw_dataset


DatasetDict({
    train: Dataset({
        features: ['title', 'abstract', 'label'],
        num_rows: 22930
    })
    test: Dataset({
        features: ['title', 'abstract', 'label'],
        num_rows: 5732
    })
})

In [ ]:
# extracting only the AI-generated samples (label = 1)

initial_abstracts = raw_dataset.filter(lambda example: example["label"] == 1)

# removing all unwanted features, keeping only "abstract"
initial_abstracts = initial_abstracts.remove_columns(["title", "label"])

initial_abstracts

DatasetDict({
    train: Dataset({
        features: ['abstract'],
        num_rows: 11465
    })
    test: Dataset({
        features: ['abstract'],
        num_rows: 2866
    })
})

In [ ]:
max_words = 250
sample_count = 900

abstracts_subset = (
    initial_abstracts["train"]
    .filter(lambda example: len(example["abstract"].split()) <= max_words)
    .shuffle(seed=42) # maintain same samples in notebook
    .select(range(sample_count))
)

In [ ]:
df_abstracts = abstracts_subset.to_pandas()
df_abstracts =  df_abstracts.replace(r'\n+|\r+', ' ', regex=True)

print(len(abstracts_subset))

df_abstracts.to_csv('ai_abstracts_900.csv', index=False)

900


In [ ]:
df_abstracts

,abstract
0,This article presents a systematic review of ...
1,This paper introduces the practical steps for...
2,This study explores the novel use of isotherm...
3,This study explores the mechanism by which he...
4,This article explores the intersection of the...
...,...
95,"This study focuses on the incidence, predict..."
96,This paper describes the successful developme...
97,This research article investigates the potent...
98,This paper explores the potential of in vitro...


In [ ]:
ai_abstract_train = []
ai_abstract_val = []

val_sample = df_abstracts.sample(n=20, random_state=42)
ai_abstract_val.append(val_sample)

train_sample = df_abstracts.drop(val_sample.index)
ai_abstract_train.append(train_sample)

# concat into dataframes for each split
df_ai_abstract_train = pd.concat(ai_abstract_train, ignore_index=True)
df_ai_abstract_val = pd.concat(ai_abstract_val, ignore_index=True)

# save to their own csv file
df_ai_abstract_train.to_csv('train_ai_abstract_samples.csv', index=False)
df_ai_abstract_val.to_csv('val_ai_abstract_samples.csv', index=False)

print(f"Validation rows: {len(df_ai_abstract_val)}")
print(f"Training rows: {len(df_ai_abstract_train)}")

Validation rows: 20
Training rows: 80


## News Articles

### AI-generated

In [6]:
# https://huggingface.co/datasets/gsingh1-py/train

raw_dataset = load_dataset('gsingh1-py/train')
raw_dataset

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['prompt', 'Human_story', 'gemma-2-9b', 'mistral-7B', 'qwen-2-72B', 'llama-8B', 'accounts/yi-01-ai/models/yi-large', 'GPT_4-o'],
        num_rows: 7321
    })
})

In [7]:
# extracting and cleaning from each model's samples

model_columns = ["gemma-2-9b", "qwen-2-72B", "llama-8B", "GPT_4-o" ]

cleaned_datasets = {}

for model in model_columns:
  if model in raw_dataset["train"].column_names:
    cleaned_datasets[model] = (
        raw_dataset.map(
            lambda example: {"sample_text": clean_article(example[model])},
            remove_columns=raw_dataset["train"].column_names
        ) # filter out the blanks (cleared out after not fitting in word count)
        .filter(lambda example: example["sample_text"] is not None and len(example["sample_text"].strip()) > 0)
    )

Map:   0%|          | 0/7321 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7321 [00:00<?, ? examples/s]

Map:   0%|          | 0/7321 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7321 [00:00<?, ? examples/s]

Map:   0%|          | 0/7321 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7321 [00:00<?, ? examples/s]

Map:   0%|          | 0/7321 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7321 [00:00<?, ? examples/s]

In [9]:
cleaned_datasets

{'gemma-2-9b': DatasetDict({
     train: Dataset({
         features: ['sample_text'],
         num_rows: 7310
     })
 }),
 'qwen-2-72B': DatasetDict({
     train: Dataset({
         features: ['sample_text'],
         num_rows: 5683
     })
 }),
 'llama-8B': DatasetDict({
     train: Dataset({
         features: ['sample_text'],
         num_rows: 7306
     })
 }),
 'GPT_4-o': DatasetDict({
     train: Dataset({
         features: ['sample_text'],
         num_rows: 7285
     })
 })}

In [10]:
gemma_articles = cleaned_datasets['gemma-2-9b']['train']
qwen_articles = cleaned_datasets['qwen-2-72B']['train']
llama_articles = cleaned_datasets['llama-8B']['train']
gpt_articles = cleaned_datasets['GPT_4-o']['train']

In [11]:
gemma_train_split, gemma_val_split = split_subset(gemma_articles, 225, 50, 250, train_split=1)
qwen_train_split, qwen_val_split = split_subset(qwen_articles, 225, 50, 250, train_split=1)
llama_train_split, llama_val_split = split_subset(llama_articles, 225, 50, 250, train_split=1)
gpt_train_split, gpt_val_split = split_subset(gpt_articles, 225, 50, 250, train_split=1)

Filter:   0%|          | 0/7310 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5683 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7306 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7285 [00:00<?, ? examples/s]

In [14]:
ai_articles_train = [gemma_train_split, qwen_train_split, llama_train_split, gpt_train_split]

# concat
df_ai_articles_train = pd.concat(ai_articles_train, ignore_index=True)

# save to their own csv file
df_ai_articles_train.to_csv('ai_articles_900.csv', index=False)

print(len(df_ai_articles_train))

900


# **Compilation**

## Training
 Set

In [ ]:
path = r"C:\Users\kacey\OneDrive\IAT360\NLP_Project\generated_samples\*.csv"
ai_samples_csv = glob.glob(path)

ai_email_train = []
ai_email_val = []

for file in ai_samples_csv:
  df = pd.read_csv(file)

  # 5 random rows
  val_sample = df.sample(n=5, random_state=42)
  ai_email_val.append(val_sample)

  # rest to training
  train_sample = df.drop(val_sample.index)
  ai_email_train.append(train_sample)

# concat
df_ai_email_train = pd.concat(ai_email_train, ignore_index=True)
df_ai_email_val = pd.concat(ai_email_val, ignore_index=True)

# save to their own csv file
df_ai_email_train.to_csv('train_ai_email_samples.csv', index=False)
df_ai_email_val.to_csv('val_ai_email_samples.csv', index=False)

print(f"Validation rows: {len(df_ai_email_val)}")
print(f"Training rows: {len(df_ai_email_train)}")


In [15]:
# load saved category subsets into dfs
df_ai_email_samples = pd.read_csv(r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\generated_samples\per category (900)\ai_emails_900.csv")
df_ai_abstract_samples = pd.read_csv(r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\generated_samples\per category (900)\ai_abstracts_900.csv")
df_ai_article_samples = pd.read_csv(r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\generated_samples\per category (900)\ai_articles_900.csv")

# add sub-type labels before compilation
df_ai_email_samples["sub-type"] = "email"
df_ai_abstract_samples["sub-type"] = "abstracts"
df_ai_article_samples["sub-type"] = "news"

# concat into one set and add their AI label (these ones are all generated samples)
ai_formal_samples=[df_ai_email_samples, df_ai_abstract_samples, df_ai_article_samples]

df_ai_formal_samples =  pd.concat(ai_formal_samples, ignore_index=True)
df_ai_formal_samples["AI_label"] = 1

# export to csv
df_ai_formal_samples.to_csv("ai_formal_samples_2700.csv", index=False)

In [35]:
# load saved category subsets into dfs
df_human_email_samples = pd.read_csv(r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\written_samples\human_emails_900.csv")
df_human_abstract_samples = pd.read_csv(r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\written_samples\human_abstracts_900.csv")
df_human_article_samples = pd.read_csv(r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\written_samples\human_articles_900.csv")

# add sub-type + AI label if needed
df_human_email_samples["sub-type"] = "email"
df_human_email_samples["AI_label"] = 0

# concat
human_formal_samples=[df_human_email_samples, df_human_abstract_samples, df_human_article_samples]
df_human_formal_samples =  pd.concat(human_formal_samples, ignore_index=True)

# export to csv
df_human_formal_samples.to_csv("human_formal_samples_2700.csv", index=False)

In [36]:
h_path = r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\per class (2700)\human_formal_samples_2700.csv"
h_df = pd.read_csv(h_path)

ai_path = r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\per class (2700)\ai_formal_samples_2700.csv"
ai_df = pd.read_csv(ai_path)

df_dataset = pd.concat([h_df, ai_df], ignore_index=True)
print(df_dataset)

df_dataset.to_csv("formal_dataset_5400.csv", index=False)


                                            sample_text sub-type  AI_label
0     Louise,\n\nWe are happy to do this.\n\nShall w...    email         0
1     Per Pam, the person we should be talking to is...    email         0
2     Louise,\n\nAttached is the list of Top 50 coun...    email         0
3     Sounds great. Just check with Audrey (3-5849) ...    email         0
4     Sally\n\nProposed list - of all potential cand...    email         0
...                                                 ...      ...       ...
5395  A Moment of UnitySanders and Biden Discuss Key...     news         1
5396  In Defense of WineRediscovering the Timeless E...     news         1
5397  HeadlineCruz Courts Establishment Support on T...     news         1
5398  Protesters Will Call for Higher Wages and Unio...     news         1
5399  City University of New York Faces Criticism fo...     news         1

[5400 rows x 3 columns]


## Validation Set

### AI-generated

In [37]:
# Load the raw dataset (https://huggingface.co/datasets/artfultom/ivypanda-llm-generated-essays)

raw_dataset = load_dataset('artfultom/ivypanda-llm-generated-essays', 'one prompt')
raw_dataset


DatasetDict({
    train: Dataset({
        features: ['title', 'prompt', 'text', 'model'],
        num_rows: 22492
    })
})

In [38]:
mistral_essays = raw_dataset.filter(lambda example: example["model"] == "mistral-7b-instruct-v0.2.Q5_K_M")
llama_essays = raw_dataset.filter(lambda example: example["model"] == "llama-3-13B-Instruct-v0.1.Q5_K_M")
deepseek_essays = raw_dataset.filter(lambda example: example["model"] == "deepseek-chat")

In [39]:
mistral_cleaned = (
    mistral_essays.map(
            lambda example: {"sample_text": clean_article(example["text"])},
            remove_columns=mistral_essays["train"].column_names
        ) # filter out the blanks (cleared out after not fitting in word count)
        .filter(lambda example: example["sample_text"] is not None and len(example["sample_text"].strip()) > 0)
    )

llama_cleaned = (
    llama_essays.map(
            lambda example: {"sample_text": clean_article(example["text"])},
            remove_columns=llama_essays["train"].column_names
        ) # filter out the blanks (cleared out after not fitting in word count)
        .filter(lambda example: example["sample_text"] is not None and len(example["sample_text"].strip()) > 0)
    )

deepseek_cleaned = (
    deepseek_essays.map(
            lambda example: {"sample_text": clean_article(example["text"])},
            remove_columns=deepseek_essays["train"].column_names
        ) # filter out the blanks (cleared out after not fitting in word count)
        .filter(lambda example: example["sample_text"] is not None and len(example["sample_text"].strip()) > 0)
    )


# create subsets with select samples for validation

mistral_samples, blank = split_subset(mistral_cleaned['train'], 135, 50, 250, train_split=1)
llama_samples, blank = split_subset(llama_cleaned['train'], 135, 50, 250, train_split=1)
deepseek_samples, blank = split_subset(deepseek_cleaned['train'], 135, 50, 250, train_split=1)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2993 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2993 [00:00<?, ? examples/s]

Map:   0%|          | 0/8999 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8999 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4495 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2993 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8685 [00:00<?, ? examples/s]

In [40]:
llm_essays = [mistral_samples, llama_samples, deepseek_samples]
df_llm_essays = pd.concat(llm_essays, ignore_index=True)

df_llm_essays["sub-type"] = "essay"
df_llm_essays

,sample_text,sub-type
0,In the complex and intricate web of religious ...,essay
1,In DreamsSurrealism and the Human Condition Su...,essay
2,"In the current societal discourse, vaccine man...",essay
3,"Afghanistan, a landlocked country located at t...",essay
4,The field of behavior analysis has gained sign...,essay
...,...,...
400,The assessment of personality represents a cor...,essay
401,The world of business is not a sterile laborat...,essay
402,The equitable and efficient distribution of va...,essay
403,The image of a nurse is often confined to the ...,essay


In [41]:
# path = r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\validation\kimi-2.6\*.csv"
# kimi_samples_csv = glob.glob(path)

# kimi_samples = []

#for file in kimi_samples_csv:
  #df = pd.read_csv(file)
  #kimi_samples.append(df)

# concat
#df_kimi_samples= pd.concat(kimi_samples, ignore_index=True)
df_kimi_samples = pd.read_csv(r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\validation\kimi_formal_samples_135.csv")

print(len(df_kimi_samples))


135


In [42]:
#path = r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\validation\meta-AI\*.csv"
#metaAI_samples_csv = glob.glob(path)

#metaAI_samples = []

#for file in metaAI_samples_csv:
  #df = pd.read_csv(file)
  #metaAI_samples.append(df)

# concat
#df_metaAI_samples= pd.concat(metaAI_samples, ignore_index=True)
df_metaAI_samples = pd.read_csv(r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\validation\metaAI_formal_samples_135.csv")


print(len(df_metaAI_samples))


135


In [43]:
kimi_subtypes = []

i=0
while i < 135:
  if 0 <= i <= 46:
    kimi_subtypes.append("nonfiction")
  elif 47 <= i <= 91:
    kimi_subtypes.append("business")
  else:
    kimi_subtypes.append("letter")
  i+= 1

metaAI_subtypes = []
i=0
while i < 135:
  if 0 <= i <= 46:
    metaAI_subtypes.append("business")
  elif 47 <= i <= 91:
    metaAI_subtypes.append("letter")
  else:
    metaAI_subtypes.append("nonfiction")
  i+= 1

print(kimi_subtypes)
print(metaAI_subtypes)

['nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'nonfiction', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'business', 'busi

In [44]:
df_kimi_samples["sub-type"] = kimi_subtypes
df_metaAI_samples["sub-type"] = metaAI_subtypes

In [46]:
AI_formal_val = [df_llm_essays, df_kimi_samples, df_metaAI_samples]

df_AI_formal_val = pd.concat(AI_formal_val, ignore_index=True)
df_AI_formal_val["AI_label"] = 1

df_AI_formal_val.to_csv("AI_formal_val_675.csv", index=False)

In [47]:
word_counts = df_AI_formal_val["sample_text"].apply(lambda t: len(t.split()))
print(word_counts.describe())

count    675.000000
mean     154.660741
std       64.281827
min       50.000000
25%       92.000000
50%      161.000000
75%      216.000000
max      250.000000
Name: sample_text, dtype: float64


## Full Set

In [58]:
human_formal_val = pd.read_csv(r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\validation\human_formal_val_675.csv")
print(human_formal_val['sub-type'].value_counts())

AI_formal_val = pd.read_csv(r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\validation\AI_formal_val_675.csv")
print(AI_formal_val['sub-type'].value_counts())

formal_val = pd.concat([human_formal_val, AI_formal_val], ignore_index=True)
formal_val.to_csv("formal_val_1350.csv", index=False)

formal_val

sub-type
letter        560
nonfiction     75
business       26
essay          14
Name: count, dtype: int64
sub-type
essay         405
business       92
nonfiction     90
letter         88
Name: count, dtype: int64


,sample_text,sub-type,AI_label
0,Even though lots of students like to work at h...,letter,0
1,Fresh air we're use to being home all day and ...,nonfiction,0
2,Many people believe that self-esteem comes fro...,letter,0
3,"Clearly, everyone should always aim high and a...",letter,0
4,I really think that can be really stressed for...,letter,0
...,...,...,...
1345,Cryptography is the discipline concerned with ...,nonfiction,1
1346,Cultural anthropology investigates the diversi...,nonfiction,1
1347,The solar system consists of the Sun and the g...,nonfiction,1
1348,Public health is defined as the organized effo...,nonfiction,1


In [59]:
word_counts = formal_val["sample_text"].apply(lambda t: len(t.split()))
print(word_counts.describe())

count    1350.000000
mean      196.757778
std        63.684916
min        48.000000
25%       157.000000
50%       228.000000
75%       248.000000
max       250.000000
Name: sample_text, dtype: float64


In [17]:
# 5400 train samples
train_path = r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\formal_train_5400.csv"

# 1350 external val samples
val_path = r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\formal_val_1350.csv"

formal_splits = {
    "train": train_path,
    "val": val_path
}

# Load as a dataset
full_dataset = load_dataset("csv", data_files=formal_splits)

full_dataset

DatasetDict({
    train: Dataset({
        features: ['sample_text', 'sub-type', 'AI_label'],
        num_rows: 5400
    })
    val: Dataset({
        features: ['sample_text', 'sub-type', 'AI_label'],
        num_rows: 1350
    })
})

## **Emergency Clean**

In [29]:
path = r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\written_samples\human_articles_900.csv"
df = pd.read_csv(path)

word_counts = df["sample_text"].apply(lambda t: len(t.split()))
print(word_counts.describe())

count      900.000000
mean      1027.922222
std       1186.615377
min         40.000000
25%        475.000000
50%        748.000000
75%       1161.250000
max      17466.000000
Name: sample_text, dtype: float64


In [30]:
df["sample_text"] = df["sample_text"].apply(clean_CNN)

word_counts = df["sample_text"].apply(lambda t: len(t.split()))
print(word_counts.describe())

count    900.000000
mean     235.397778
std       17.694591
min       40.000000
25%      230.000000
50%      239.000000
75%      246.000000
max      250.000000
Name: sample_text, dtype: float64


In [34]:
df.to_csv("human_articles_900_cleaned.csv", index=False)

In [55]:
path = r"C:\Users\kacey\OneDrive - Simon Fraser University (1sfu)\IAT360\NLP Project\dataset\validation\human_formal_val_675.csv"
df = pd.read_csv(path)

word_counts = df["sample_text"].apply(lambda t: len(t.split()))
print(word_counts.describe())

count     675.000000
mean      440.346667
std       194.307188
min        48.000000
25%       294.500000
50%       406.000000
75%       542.000000
max      1170.000000
Name: sample_text, dtype: float64


In [56]:
df["sample_text"] = df["sample_text"].apply(clean_CNN)

word_counts = df["sample_text"].apply(lambda t: len(t.split()))
print(word_counts.describe())

count    675.000000
mean     238.854815
std       20.876341
min       48.000000
25%      237.000000
50%      248.000000
75%      250.000000
max      250.000000
Name: sample_text, dtype: float64


In [57]:
df.to_csv("human_formal_val_675_cleaned.csv", index=False)